# KEGG pathway ORA across all 16 drug contexts → tensor

Builds a **(drug × perturbation × pathway)** tensor of pathway-enrichment
scores from FDR-filtered DE genes.  Same ORA logic as notebook 15, looped
over every drug, with results stacked into a single labeled tensor.

Saved artifacts (in `ChemoGeneticScreens/PathwayORA/KEGG/`):

1. **`KEGG_ORA_tensor.h5`** — HDF5 with three (drug × pert × pathway)
   tensors for the headline `direction = 'any'`:
     - `neg_log10_q`  — primary enrichment score, `-log10(q_bh)`
     - `q_value`      — raw BH q-value
     - `odds_ratio`   — observed / expected overlap ratio
   Axis labels are stored as datasets `drugs`, `perturbations`, `pathways`.
2. **`KEGG_ORA_all_drugs_long.parquet`** — long-format table, one row per
   `(drug, perturbation, pathway, direction ∈ {any, up, down})` for any
   downstream analysis that needs direction-resolved scores.

In [1]:
import time
import h5py
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests

## Paths and parameters

In [2]:
PROJECT_DIR = Path('/home/beraslan/Projects/ChemoGeneticScreens')
PM_DIR      = PROJECT_DIR / 'PosteriorMeanMatrices'
FDR_DIR     = PROJECT_DIR / 'FDR_matrices'
OUT_DIR     = PROJECT_DIR / 'PathwayORA' / 'KEGG'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# MSigDB v2024.1.Hs splits KEGG into two complementary, non-overlapping GMTs:
#   - kegg_legacy  : 186 classical pathway maps (same as v7.5.1)
#   - kegg_medicus : 658 newer KEGG MEDICUS network/disease modules
# Union ~844 pathways before universe + size filtering.
KEGG_DIR = Path('/home/beraslan/Projects/ModuleFinder/MuVI/msigdb')
KEGG_GMTS = [
    KEGG_DIR / 'c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt',
    KEGG_DIR / 'c2.cp.kegg_medicus.v2024.1.Hs.symbols.gmt',
]

FDR_THRESHOLD     = 0.01
LFC_THRESHOLD     = 0.15   # |posterior-mean logFC| must exceed this in addition to FDR
MIN_PATHWAY_SIZE  = 10
MAX_PATHWAY_SIZE  = 500
MIN_SIG_PER_PERT  = 5

# All 16 contexts — derived from filenames
DRUGS = sorted(p.name.replace('PosteriorMean_matrix_', '').replace('.csv', '')
               for p in PM_DIR.glob('PosteriorMean_matrix_*.csv'))
print(f'{len(DRUGS)} drug contexts:')
for d in DRUGS:
    print(f'  {d}')

16 drug contexts:
  AR-A014418
  AZD4573
  Bisindolylmaleimide-I
  CHIR-98014
  DG-172
  DMSO_round2
  DMSO_round2_batch2
  JTE-607
  LDN-193189
  LY2090314
  Lexibulin
  NSC95397
  PP121
  Romidepsin
  Stattic
  VX-11e


## Helpers (parse GMT, load drug, build membership, vectorised ORA)

In [3]:
def parse_gmt(path):
    out = {}
    with open(path) as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 3:
                continue
            name, _url, *genes = parts
            genes = {g for g in genes if g}
            if genes:
                out[name] = genes
    return out


def load_drug(drug):
    pm  = pd.read_csv(PM_DIR  / f'PosteriorMean_matrix_{drug}.csv', index_col=0)
    fdr = pd.read_csv(FDR_DIR / f'{drug}_FDRs.csv',                index_col=0)
    common_p = pm.index.intersection(fdr.index)
    common_g = pm.columns.intersection(fdr.columns)
    return pm.loc[common_p, common_g], fdr.loc[common_p, common_g]


def ora_pq_matrices(S, M, K_sizes, N, min_sig=5):
    """Vectorised hypergeometric ORA — returns (p_mat, q_mat, k_mat, odds, kept_mask).

    Rows where the perturbation has < min_sig sig genes are filled with NaN
    so the output shape always equals S.shape[0] x M.shape[0].
    """
    n_p     = S.sum(axis=1)
    n_perts = S.shape[0]
    P       = M.shape[0]
    p_mat = np.full((n_perts, P), np.nan, dtype=np.float64)
    q_mat = np.full((n_perts, P), np.nan, dtype=np.float64)
    k_mat = np.zeros((n_perts, P), dtype=np.int32)
    odds  = np.full((n_perts, P), np.nan, dtype=np.float64)

    keep = n_p >= min_sig
    if not keep.any():
        return p_mat, q_mat, k_mat, odds, keep

    S_k = S[keep]
    n_k = n_p[keep]
    k_kept = (S_k.astype(np.int32) @ M.astype(np.int32).T)  # (m, P)
    K_b = np.broadcast_to(K_sizes,         k_kept.shape)
    n_b = np.broadcast_to(n_k[:, None],    k_kept.shape)
    p_kept = hypergeom.sf(k_kept - 1, N, K_b, n_b)

    q_kept = np.empty_like(p_kept)
    for i in range(p_kept.shape[0]):
        _, q, _, _ = multipletests(p_kept[i], method='fdr_bh')
        q_kept[i] = q
    expected = (K_b * n_b) / N
    odds_kept = np.where(expected > 0, k_kept / expected, np.nan)

    p_mat[keep] = p_kept
    q_mat[keep] = q_kept
    k_mat[keep] = k_kept
    odds[keep]  = odds_kept
    return p_mat, q_mat, k_mat, odds, keep

## Load KEGG once + define the universe / membership matrix

The universe (`tested ∩ KEGG`) is fixed across drugs because the response-gene
columns are the same across all matrices.  We load one drug to get the column
set and reuse the membership matrix for every drug.

In [4]:
kegg_raw = {}
for gmt in KEGG_GMTS:
    parsed = parse_gmt(gmt)
    kegg_raw.update(parsed)
    print(f'  {gmt.name}: {len(parsed)} pathways')
print(f'Total KEGG pathways loaded: {len(kegg_raw)}')

# Probe one drug for the gene-column set (assumed identical across drugs)
_pm_probe, _ = load_drug(DRUGS[0])
tested_genes = list(_pm_probe.columns)
kegg_union   = set().union(*kegg_raw.values())
universe     = [g for g in tested_genes if g in kegg_union]
gene_to_idx  = {g: i for i, g in enumerate(universe)}
N_UNIV       = len(universe)
del _pm_probe

kegg = {n: g & set(universe) for n, g in kegg_raw.items()}
kegg = {n: g for n, g in kegg.items() if MIN_PATHWAY_SIZE <= len(g) <= MAX_PATHWAY_SIZE}
pathway_names = sorted(kegg)
P = len(pathway_names)
M = np.zeros((P, N_UNIV), dtype=bool)
for i, name in enumerate(pathway_names):
    idx = [gene_to_idx[g] for g in kegg[name]]
    M[i, idx] = True
K_sizes = M.sum(axis=1)
print(f'Universe: {N_UNIV} genes, pathways kept after size filter: {P}')

  c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt: 186 pathways
  c2.cp.kegg_medicus.v2024.1.Hs.symbols.gmt: 658 pathways
Total KEGG pathways loaded: 844
Universe: 5658 genes, pathways kept after size filter: 568


## Run ORA for every drug × direction

Per drug we keep the per-perturbation `(P,)` row of `q_bh`/`p`/`odds_ratio`
for each of the three directions, plus the perturbation index for that drug.
Across drugs perturbations may differ slightly; we compute the **union** axis
afterwards and pad missing rows with NaN.

In [5]:
DIRECTIONS = ('any',)   # tuple — only the 'any' direction is used downstream
per_drug = {}                                             # drug -> {direction -> {p,q,k,odds}, perts}
all_perts = []                                            # ordered union of perturbations
seen_perts = set()

for drug in DRUGS:
    t0 = time.time()
    pm, fdr = load_drug(drug)
    perts = list(pm.index)
    for p in perts:
        if p not in seen_perts:
            all_perts.append(p)
            seen_perts.add(p)

    lfc_u = pm[universe].values
    fdr_u = fdr[universe].values
    sig_any = (fdr_u < FDR_THRESHOLD) & (np.abs(lfc_u) > LFC_THRESHOLD)

    drug_res = {'perts': perts}
    for direction, S in zip(DIRECTIONS, (sig_any,)):
        p_mat, q_mat, k_mat, odds, _ = ora_pq_matrices(S, M, K_sizes, N_UNIV,
                                                       min_sig=MIN_SIG_PER_PERT)
        drug_res[direction] = {'p': p_mat, 'q': q_mat, 'k': k_mat, 'odds': odds}
    per_drug[drug] = drug_res
    print(f'  {drug:30s}  {len(perts):>5d} perts  ({time.time()-t0:5.1f}s)')

print(f'\nUnion of perturbations across drugs: {len(all_perts)}')

  AR-A014418                       2294 perts  ( 28.7s)
  AZD4573                          2248 perts  ( 26.8s)
  Bisindolylmaleimide-I            2259 perts  ( 27.3s)
  CHIR-98014                       2226 perts  ( 26.6s)
  DG-172                           2307 perts  ( 29.1s)
  DMSO_round2                      2212 perts  ( 25.7s)
  DMSO_round2_batch2               2225 perts  ( 25.8s)
  JTE-607                          2340 perts  ( 30.2s)
  LDN-193189                       2269 perts  ( 27.4s)
  LY2090314                        2201 perts  ( 25.9s)
  Lexibulin                        2296 perts  ( 28.1s)
  NSC95397                         2083 perts  ( 26.8s)
  PP121                            2249 perts  ( 26.6s)
  Romidepsin                       2288 perts  ( 28.0s)
  Stattic                          2292 perts  ( 28.4s)
  VX-11e                           2271 perts  ( 28.9s)

Union of perturbations across drugs: 2447


  Bisindolylmaleimide-I            2259 perts  ( 67.2s)


  CHIR-98014                       2226 perts  ( 66.8s)


  DG-172                           2307 perts  ( 75.7s)


  DMSO_round2                      2212 perts  ( 61.3s)


...

  NSC95397                         2083 perts  ( 67.1s)


  PP121                            2249 perts  ( 65.5s)


  Romidepsin                       2288 perts  ( 66.8s)


  Stattic                          2292 perts  ( 73.0s)


  VX-11e                           2271 perts  ( 69.3s)

Union of perturbations across drugs: 2447


## Stack into the (drug × perturbation × pathway) tensor

Headline tensor is `direction = 'any'`, score = `-log10(q_bh)` (clipped at
1e-300 to avoid `inf`).  For each drug, perturbations missing in that drug
stay as `NaN` so downstream code can mask them.

In [6]:
n_drugs, n_perts, n_paths = len(DRUGS), len(all_perts), P
pert_to_global = {p: i for i, p in enumerate(all_perts)}

neg_log10_q = np.full((n_drugs, n_perts, n_paths), np.nan, dtype=np.float32)
q_value     = np.full_like(neg_log10_q, np.nan)
odds_ratio  = np.full_like(neg_log10_q, np.nan)

for d_idx, drug in enumerate(DRUGS):
    drug_res = per_drug[drug]
    perts    = drug_res['perts']
    rows     = [pert_to_global[p] for p in perts]
    q        = drug_res['any']['q']
    odds     = drug_res['any']['odds']
    q_value[d_idx, rows, :]     = q.astype(np.float32)
    odds_ratio[d_idx, rows, :]  = odds.astype(np.float32)
    neg_log10_q[d_idx, rows, :] = (-np.log10(np.clip(q, 1e-300, 1.0))).astype(np.float32)

print(f'Tensor shape: ({n_drugs}, {n_perts}, {n_paths})')
print(f'Bytes per tensor (float32): {neg_log10_q.nbytes / 1e6:.1f} MB')
print(f'NaN fraction in -log10(q): {np.isnan(neg_log10_q).mean()*100:.1f}%')
print(f'Significant cells (q<0.1): {(q_value < 0.1).sum():,}'
      f'  ({(q_value < 0.1).sum() / np.isfinite(q_value).sum() * 100:.2f}% of finite cells)')

Tensor shape: (16, 2447, 568)
Bytes per tensor (float32): 89.0 MB
NaN fraction in -log10(q): 62.3%
Significant cells (q<0.1): 117,813  (1.41% of finite cells)


## Save

1. HDF5 with the 3D tensors and axis labels.
2. Long-format Parquet with all three directions for downstream filtering.

In [7]:
h5_path = OUT_DIR / 'KEGG_ORA_tensor.h5'
with h5py.File(h5_path, 'w') as h:
    h.create_dataset('neg_log10_q', data=neg_log10_q, compression='gzip', compression_opts=4)
    h.create_dataset('q_value',     data=q_value,     compression='gzip', compression_opts=4)
    h.create_dataset('odds_ratio',  data=odds_ratio,  compression='gzip', compression_opts=4)
    str_dt = h5py.string_dtype(encoding='utf-8')
    h.create_dataset('drugs',         data=np.array(DRUGS,         dtype=object), dtype=str_dt)
    h.create_dataset('perturbations', data=np.array(all_perts,     dtype=object), dtype=str_dt)
    h.create_dataset('pathways',      data=np.array(pathway_names, dtype=object), dtype=str_dt)
    h.attrs['fdr_threshold']    = FDR_THRESHOLD
    h.attrs['lfc_threshold']    = LFC_THRESHOLD
    h.attrs['min_pathway_size'] = MIN_PATHWAY_SIZE
    h.attrs['max_pathway_size'] = MAX_PATHWAY_SIZE
    h.attrs['min_sig_per_pert'] = MIN_SIG_PER_PERT
    h.attrs['direction']        = 'any'
    h.attrs['kegg_gmts']        = ';'.join(str(p) for p in KEGG_GMTS)
print(f'Saved {h5_path}  ({h5_path.stat().st_size/1e6:.1f} MB)')

Saved /home/beraslan/Projects/ChemoGeneticScreens/PathwayORA/KEGG/KEGG_ORA_tensor.h5  (18.1 MB)


In [8]:
long_rows = []
for d_idx, drug in enumerate(DRUGS):
    drug_res = per_drug[drug]
    perts    = drug_res['perts']
    rows     = [pert_to_global[p] for p in perts]
    for direction in DIRECTIONS:
        d = drug_res[direction]
        valid = ~np.isnan(d['q'])              # drop perturbations with too few sig genes
        valid_rows, valid_cols = np.where(valid)
        long_rows.append(pd.DataFrame({
            'drug':         drug,
            'perturbation': [perts[i]                   for i in valid_rows],
            'pathway':      [pathway_names[j]           for j in valid_cols],
            'direction':    direction,
            'k_overlap':    d['k'][valid_rows, valid_cols].astype(np.int32),
            'K_size':       K_sizes[valid_cols].astype(np.int32),
            'p_value':      d['p'][valid_rows, valid_cols].astype(np.float32),
            'q_bh':         d['q'][valid_rows, valid_cols].astype(np.float32),
            'odds_ratio':   d['odds'][valid_rows, valid_cols].astype(np.float32),
        }))
long_df = pd.concat(long_rows, ignore_index=True)
long_path = OUT_DIR / 'KEGG_ORA_all_drugs_long.parquet'
long_df.to_parquet(long_path, index=False)
print(f'Saved {long_path}  ({long_path.stat().st_size/1e6:.1f} MB, {len(long_df):,} rows)')

Saved /home/beraslan/Projects/ChemoGeneticScreens/PathwayORA/KEGG/KEGG_ORA_all_drugs_long.parquet  (28.7 MB, 8,378,568 rows)


## Quick sanity / shape check

Per-drug count of significantly-enriched (perturbation, pathway) cells
(`q_bh < 0.1`) at the `direction = 'any'` slice.

In [9]:
sig_per_drug = []
for d_idx, drug in enumerate(DRUGS):
    sl = q_value[d_idx]                              # (n_perts, n_paths)
    finite = np.isfinite(sl)
    n_sig = int((sl < 0.1).sum())
    n_total = int(finite.sum())
    sig_per_drug.append({
        'drug':        drug,
        'tested_cells': n_total,
        'sig_cells':   n_sig,
        'pct_sig':     100.0 * n_sig / max(n_total, 1),
    })
print(pd.DataFrame(sig_per_drug).to_string(index=False))

                 drug  tested_cells  sig_cells  pct_sig
           AR-A014418        550392      11295 2.052174
              AZD4573        484504       6338 1.308142
Bisindolylmaleimide-I        499840       6158 1.231994
           CHIR-98014        598104       4183 0.699377
               DG-172        543008      12143 2.236247
          DMSO_round2        477688       4364 0.913567
   DMSO_round2_batch2        467464       4460 0.954084
              JTE-607        560048      14552 2.598349
           LDN-193189        523128       6409 1.225130
            LY2090314        558344       3623 0.648883
            Lexibulin        526536       8787 1.668832
             NSC95397        499840       7414 1.483275
                PP121        477120       5743 1.203680
           Romidepsin        516312       7765 1.503936
              Stattic        511768       7817 1.527450
               VX-11e        584472       6762 1.156942
